# Dataset

1. Get Token: <https://huggingface.co/docs/hub/en/security-tokens>

2. Run `hf auth login` in your terminal

3. Make sure you start downloading ImageNet before everything else, `python data/download.py`. This script will take around 2 hours to download and save 250,000 images.

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T
from datasets import load_dataset
from torch.utils.data import DataLoader

In [ ]:
def display_samples(dataset):
    # Get the class label and sample from the dataset
    class_label = dataset["train"].features["label"]
    num_classes = class_label.num_classes
    print(f"Number of classes: {num_classes}")
    print(f"Class names: {class_label.names}")

    # Display one sample from each class
    # Limit to 25 classes for display
    num_classes = min(num_classes, 25)
    indexes, labels = [], []
    for i, sample in enumerate(dataset["train"]):
        label = sample["label"]
        if label not in labels:
            indexes.append(i)
            labels.append(label)
        if len(indexes) >= num_classes:
            break

    # Set up the figure and axes
    col = 5
    row = num_classes // col + (num_classes % col > 0)
    fig, axes = plt.subplots(row, col, figsize=(col * 5, row * 5))
    axes = axes.flatten()
    for i, idx in enumerate(indexes):
        sample = dataset["train"][idx]
        axes[i].imshow(sample["image"])
        axes[i].set_title(class_label.int2str(sample["label"]))
        axes[i].axis("off")

    plt.show()

## ImageNet-1K

HuggingFace: <https://huggingface.co/datasets/ILSVRC/imagenet-1k>

Homepage: <https://www.image-net.org/>

ILSVRC 2012, commonly known as 'ImageNet' is an image dataset organized according to the WordNet hierarchy. Each meaningful concept in WordNet, possibly described by multiple words or word phrases, is called a "synonym set" or "synset". There are more than 100,000 synsets in WordNet, majority of them are nouns (80,000+). ImageNet aims to provide on average 1000 images to illustrate each synset. Images of each concept are quality-controlled and human-annotated.

💡 This dataset provides access to ImageNet (ILSVRC) 2012 which is the most commonly used **subset** of ImageNet. This dataset spans 1000 object classes and contains 1,281,167 training images, 50,000 validation images and 100,000 test images.

In [ ]:
tiny_imgnet = load_dataset("zh-plus/tiny-imagenet")
tiny_imgnet

In [ ]:
display_samples(tiny_imgnet)

## With PyTorch

1. <https://huggingface.co/docs/datasets/use_with_pytorch>
2. <https://docs.pytorch.org/vision/stable/transforms.html#>

In [ ]:
ds = tiny_imgnet.with_format("torch")["train"]

imgnet_mean = [0.485, 0.456, 0.406]
imanet_std = [0.229, 0.224, 0.225]
transform = T.Compose(
    [
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=imgnet_mean, std=imanet_std),
    ]
)


def transform_batch(batch):
    batch["image"] = [transform(image) for image in batch["image"]]
    return batch


ds.set_transform(transform_batch)
ds

In [ ]:
dataloader = DataLoader(ds, batch_size=4, shuffle=True)
for batch in dataloader:
    images, labels = batch["image"], batch["label"]
    print(f"Batch of images shape: {images.shape}")
    print(f"Batch of labels shape: {labels.shape}")
    break